# Lab 1: Fashion-MNIST Classification - Presentation Notebook
## Part B: Training, Evaluation, Visualization, and Experiments

This notebook demonstrates Member B's pipeline components by calling modular functions directly.
- **Module `train`**: Model training, sample-weighted loss, checkpointing
- **Module `evaluate`**: Model evaluation, confusion matrix, per-class accuracy
- **Module `visualize`**: Loss curves, sample predictions grid, confusion matrix heatmap
- **Module `experiment`**: Multi-configuration ablation study, single test evaluation

In [ ]:
import sys
from pathlib import Path
import torch
import pandas as pd
from IPython.display import Image, display

# Ensure repo root is on sys.path
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from shared.paths import raw_data_dir, artifact_dir
from shared.seed import set_seed
from labs.lab1.preprocessing.preprocessing import load_csv, load_split
from labs.lab1.features.features import compute_mean_std, build_transform, FashionDataset, make_loaders
from labs.lab1.model.model import MLP, count_parameters
from labs.lab1.train.train import train_model, load_checkpoint
from labs.lab1.evaluate.evaluate import evaluate, confusion_matrix, per_class_accuracy
from labs.lab1.visualize.visualize import plot_loss_curves, show_predictions, plot_confusion_matrix
from labs.lab1.experiment.experiment import run_experiments, save_results

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

### 1. Data Preparation
Loading pre-computed split and train-only normalization statistics.

In [ ]:
train_csv = str(raw_data_dir('lab1') / 'fashion-mnist_train.csv')
test_csv = str(raw_data_dir('lab1') / 'fashion-mnist_test.csv')
split_path = str(artifact_dir('lab1', 'splits') / 'train_val_split.json')

images, labels = load_csv(train_csv)
test_images, test_labels = load_csv(test_csv)
train_idx, val_idx = load_split(split_path)

mean, std = compute_mean_std(images[train_idx])
transform = build_transform(mean, std)

train_ds = FashionDataset(images, labels, indices=train_idx, transform=transform)
val_ds = FashionDataset(images, labels, indices=val_idx, transform=transform)
test_ds = FashionDataset(test_images, test_labels, indices=None, transform=transform)

train_loader, val_loader, test_loader = make_loaders(train_ds, val_ds, test_ds, batch_size=64)
print(f'Train samples: {len(train_ds)}, Val samples: {len(val_ds)}, Test samples: {len(test_ds)}')

### 2. Module `train`: Baseline Model Training
Training baseline MLP (784 -> 256 -> 128 -> 10) for 5 epochs with Adam optimizer.

In [ ]:
model = MLP(hidden_sizes=(256, 128), num_classes=10, dropout=0.0)
ckpt_path = str(artifact_dir('lab1', 'models') / 'mlp_baseline.pt')

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    lr=1e-3,
    device=device,
    ckpt_path=ckpt_path,
    optimizer_name='adam',
    weight_decay=0.0
)

for ep in range(len(history['train_loss'])):
    print(f'Epoch {ep+1}/5: Train Loss={history["train_loss"][ep]:.4f}, Val Loss={history["val_loss"][ep]:.4f}, Val Acc={history["val_acc"][ep]*100:.2f}%')

### 3. Module `evaluate`: Model Evaluation & Confusion Matrix
Loading saved best checkpoint and computing validation loss, accuracy, and per-class metrics.

In [ ]:
eval_model = MLP(hidden_sizes=(256, 128), num_classes=10, dropout=0.0)
best_model = load_checkpoint(eval_model, ckpt_path, device=device)

val_loss, val_acc = evaluate(best_model, val_loader, device=device)
cm = confusion_matrix(best_model, val_loader, device=device, num_classes=10)
per_class = per_class_accuracy(cm)

CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print(f'Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc*100:.2f}%')
for c, name in enumerate(CLASS_NAMES):
    print(f'  Class {c} ({name:<11}): {per_class[c]*100:5.2f}% ({cm[c, c]}/{cm[c].sum()})')

### 4. Module `visualize`: Plotting Curves, Predictions, & Heatmaps

In [ ]:
fig_dir = artifact_dir('lab1', 'figures')
loss_path = str(fig_dir / 'loss_curves.png')
pred_path = str(fig_dir / 'predictions_grid.png')
cm_path = str(fig_dir / 'confusion_matrix.png')

plot_loss_curves(history, loss_path)
show_predictions(best_model, val_loader, CLASS_NAMES, device=device, out_path=pred_path, n=16)
plot_confusion_matrix(cm, CLASS_NAMES, cm_path)

display(Image(loss_path))
display(Image(pred_path))
display(Image(cm_path))

### 5. Module `experiment`: Controlled Ablation Experiments
Comparing 4 configurations on validation data, followed by a single final test set evaluation.

In [ ]:
configs = [
    {'name': 'baseline', 'hidden_sizes': (256, 128), 'lr': 1e-3, 'dropout': 0.0, 'optimizer_name': 'adam', 'weight_decay': 0.0, 'epochs': 5, 'seed': 42},
    {'name': 'optimizer_sgd', 'hidden_sizes': (256, 128), 'lr': 0.01, 'dropout': 0.0, 'optimizer_name': 'sgd', 'weight_decay': 0.0, 'epochs': 5, 'seed': 42},
    {'name': 'larger_network', 'hidden_sizes': (512, 256, 128), 'lr': 1e-3, 'dropout': 0.0, 'optimizer_name': 'adam', 'weight_decay': 0.0, 'epochs': 5, 'seed': 42},
    {'name': 'with_dropout_reg', 'hidden_sizes': (256, 128), 'lr': 1e-3, 'dropout': 0.2, 'optimizer_name': 'adam', 'weight_decay': 1e-4, 'epochs': 5, 'seed': 42},
]

results = run_experiments(configs, (train_loader, val_loader, test_loader), device=device)
runs_csv = str(artifact_dir('lab1', 'runs') / 'experiment_results.csv')
save_results(results, runs_csv)

df_results = pd.read_csv(runs_csv)
display(df_results)

# Evaluate best model once on test set
test_loss, test_acc = evaluate(best_model, test_loader, device=device)
print(f'\nFinal Test Loss: {test_loss:.4f} | Final Test Accuracy: {test_acc*100:.2f}%')